# 举例1：大模型分析工具的调用

In [1]:
# 1、获取大模型
#导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]

# 3、因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数:convert_to_openai_function()
# 大模型（OpenAI API）看不懂 LangChain 的 Python 对象。这个函数的作用是将 Python 的工具对象转换成 OpenAI API 能识别的 JSON Schema 格式。
"""
{
  "name": "move_file",
  "description": "Move files from one location to another",
  "parameters": {
    "type": "object",
    "properties": {
      "source_path": {"type": "string"},
      "destination_path": {"type": "string"}
    },
    "required": ["source_path", "destination_path"]
  }
}
"""
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
messages = [HumanMessage(content="将文件a移动到桌面")]

# 5、调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 76, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'function_call', 'logprobs': None} id='run-3fe13ff8-4fe0-4ada-9078-3a6106454215-0' usage_metadata={'input_tokens': 76, 'output_tokens': 27, 'total_tokens': 103}


In [1]:
# 1、获取大模型
from langchain_community.tools import DuckDuckGoSearchRun  # <--- 变化点：导入搜索工具
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
# <--- 变化点：实例化搜索工具
tools = [DuckDuckGoSearchRun()]

# 3、将工具转换为OpenAI能看懂的函数定义 (JSON Schema)
# 这一步逻辑没变，它会把 DuckDuckGo 的 Python 对象变成 JSON 描述
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
# <--- 变化点：提问需要搜索的内容
messages = [HumanMessage(content="帮我查一下LangChain最新的版本号是多少")]

# 5、调用大模型（只决策，不执行）
response = chat_model.invoke(
    input=messages,
    functions=functions, # 传入函数定义
)

# 打印结果
print("=== 模型返回的决策信息 ===")
print(response.additional_kwargs)


/Users/dingchuan/Documents/Repos/ai-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== 模型返回的决策信息 ===
{'function_call': {'arguments': '{"query":"LangChain latest version number"}', 'name': 'duckduckgo_search'}, 'refusal': None}


In [2]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 定义并获取工具列表
@tool
def count_string_length(text:str) -> int:
    """计算给定走的长度"""
    return len(text.split())

tools = [count_string_length]

# 3. 转换未OpenAI的函数定义
# 这一步会自动读取函数的 docstring （注释）和参数类型 生成 Schema
functions = [convert_to_openai_function(t) for t in tools]

# 4. 获取消息列表
# <--- 变化点：用户请求计算长度
messages = [HumanMessage(content="单词 'Supercalifragilisticexpialidocious' 有多少个字母？")]

# 5. 调用大模型
response = chat_model.invoke(
    input=messages,
    functions=functions,
)

print("=== 模型返回的决策信息 ===")
print(response.additional_kwargs)

=== 模型返回的决策信息 ===
{'function_call': {'arguments': '{"text":"Supercalifragilisticexpialidocious"}', 'name': 'count_string_length'}, 'refusal': None}



## 核心知识点总结

无论你换成什么 Tool，这种实现方式的固定套路都是：

1. 准备 Tool：可以是 LangChain 内置的（如 MoveFileTool, DuckDuckGoSearchRun），也可以是你自己用 @tool 写的方法。
2. 翻译 Tool (convert_to_openai_function)：这一步必不可少。因为大模型不认识 Python 代码，它只认识 JSON 格式的函数描述（Function Schema）。
    - 它把 Python 的 def func(a: int) 翻译成 JSON 的 {"name": "func", "parameters": {"a": "integer"}}。
3. 传给模型 (functions=...)：把翻译好的 JSON 列表扔给模型。
4. 模型决策：模型根据你的 HumanMessage 和 functions 描述，判断出：“嗯，为了解决这个问题，我应该调用这个函数”。
### 注意：

再次强调，这种 functions=functions 的写法是 LangChain 较早期的底层写法。虽然它现在还能用，且非常有助于理解原理，但在生产环境中，建议逐渐过渡到 llm.bind_tools(tools) 这种更现代的写法。


In [3]:
# 1、获取大模型
from langchain_community.tools import DuckDuckGoSearchRun  # <--- 变化点：导入搜索工具
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
# <--- 变化点：实例化搜索工具
tools = [DuckDuckGoSearchRun()]

# 3、将工具转换为OpenAI能看懂的函数定义 (JSON Schema)
# 这一步逻辑没变，它会把 DuckDuckGo 的 Python 对象变成 JSON 描述
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
# <--- 变化点：提问需要搜索的内容
messages = [HumanMessage(content="帮我查一下今天是周几")]

# 5、调用大模型（只决策，不执行）
response = chat_model.invoke(
    input=messages,
    functions=functions, # 传入函数定义
)

# 打印结果
print("=== 模型返回的决策信息 ===")
print(response.additional_kwargs)


=== 模型返回的决策信息 ===
{'function_call': {'arguments': '{"query":"今天是周几"}', 'name': 'duckduckgo_search'}, 'refusal': None}


作为对比：


In [2]:
# 获取消息列表
messages = [HumanMessage(content="查询一下明天北京的天气")]

# 调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

print(response)

content='抱歉，我无法提供实时天气信息。建议您查看天气预报网站或使用天气应用程序获取最新的天气信息。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 76, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'stop', 'logprobs': None} id='run-84bd374d-d002-4006-9ace-b940e4df660e-0' usage_metadata={'input_tokens': 76, 'output_tokens': 29, 'total_tokens': 105}


通过上面两个测试发现，得到的AIMessage的核心属性如下：

1、如果分析出需要调用对应的工具：

content：信息为空。因为大模型要调用工具，所以就不会直接返回信息给用户

additional_kwargs：包含function_call字段，指明具体函数调用的参数和函数名。比如：

additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None}

2、如果分析出不需要调用对应的工具：

content：信息不为空。

additional_kwargs：不包含function_call字段








## 现代的写法

在现代 LangChain（v0.1+ 和 v0.2+）中，我们不再手动转换函数或使用 functions 参数，而是使用 bind_tools() 方法。同时，OpenAI 的 API 也已经从 functions 升级到了 tools。

这是最标准、最现代的写法：

### 核心变化点
1. 废弃：convert_to_openai_function 和 invoke(..., functions=...)。
2. 启用：llm.bind_tools(tools)。这是 LangChain 提供的统一接口，会自动处理模型特定的工具绑定逻辑（无论是 OpenAI、Anthropic 还是 Google Gemini）。
3. 结果获取：不再去 additional_kwargs 里翻找，而是直接读取 response.tool_calls 属性。

### 现代写法代码实现

In [3]:
# 1. 导入依赖
import os
import dotenv
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. 定义工具列表
tools = [MoveFileTool()]

# ==========================================
# 核心变化：使用 bind_tools 进行绑定
# ==========================================
# 这行代码自动完成了以下工作：
# 1. 将 LangChain 工具转换为 OpenAI 的 tool schema (JSON)
# 2. 将 tools 参数绑定到模型调用中
llm_with_tools = chat_model.bind_tools(tools)

# 4. 构造消息
messages = [HumanMessage(content="将文件 report.pdf 移动到 backup 文件夹")]

# 5. 调用模型
# 注意：我们调用的是绑定了工具的 llm_with_tools
response = llm_with_tools.invoke(messages)

# 6. 打印结果
print("=== 原始响应对象 ===")
print(response)

print("\n=== 提取工具调用信息 (推荐) ===")
# 现代 LangChain 会自动解析工具调用到 tool_calls 属性中，这是一个标准的列表
if response.tool_calls:
    for tool_call in response.tool_calls:
        print(f"工具名称: {tool_call['name']}")
        print(f"参数内容: {tool_call['args']}")
        print(f"调用ID:   {tool_call['id']}")
else:
    print("模型未决定调用工具")

=== 原始响应对象 ===
content='' additional_kwargs={'tool_calls': [{'id': 'call_1wUEXIvgluCmsfbd6B6BhdrM', 'function': {'arguments': '{"source_path":"report.pdf","destination_path":"backup/report.pdf"}', 'name': 'move_file'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 80, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--e1438434-77f0-436a-8364-e1b39b5764da-0' tool_calls=[{'name': 'move_file', 'args': {'source_path': 'report.pdf', 'destination_path': 'backup/report.pdf'}, 'id': 'call_1wUEXIvgluCmsfbd6B6BhdrM', 'type': 'tool_call'}] usage_metadata={'input_tokens': 80, 'output_tokens': 24, 'total_tokens': 104}

===

### 为什么这种写法更好？
1. 标准化 (Standardization)：
response.tool_calls 是 LangChain 定义的标准属性。无论底层用的是 OpenAI、Claude 还是 Mistral，你获取工具参数的方式都是一样的，不需要去解析各个模型特有的 JSON 结构。
2. 简洁 (Simplicity)：
bind_tools() 一行代码替代了之前的 convert_to_openai_function 循环和手动传参，代码更干净。
3. 未来兼容性 (Future Proof)：
OpenAI 官方推荐使用 tools 而不是 functions（functions 已被标记为过时）。bind_tools 会自动使用 OpenAI 最新的 API 规范。


## 进阶： 如果你想直接执行工具

在现代写法中，如果你想让工具真的跑起来，通常会结合 LangGraph（LangChain 的新一代编排库）或者使用内置的 create_tool_calling_agent。


最简单的自动执行方式如下：

In [ ]:
from langchain.agents import create_tool_calling_agent,AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 定义 Prompt
# 定义 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"), # 必须包含这个占位符
])

# 创建 Agent (大脑)
agent = create_tool_calling_agent(chat_model, tools, prompt)

# 创建 执行器 (手脚)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 运行 (自动 决策->执行->反馈 循环)
agent_executor.invoke({"input": "把 a.txt 移到 b 文件夹"})

# 举例2：如何调用具体大模型分析出来的工具

说明：

1、大模型与Agent的核心区别：是否涉及到工具的调用

2、针对于大模型：仅能分析出要调用的工具，但是此工具（或函数）不能真正的执行

   针对于Agent:除了分析出要调用的工具之外，还可以执行具体的工具（或函数）

In [8]:
# 1、获取大模型
#导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]

# 3、因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数:convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
messages = [HumanMessage(content="将当前目录下的文件a.txt移动到C:\\Users\\shkst\\Desktop")]

# 5、调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

# print(response)

步骤1：分析下要调用哪个工具或函数

In [11]:
import json

if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具：{tool_name} \n 参数：{tool_args}")

else:
    print(f"模型回复：{response.content}")

调用工具：move_file 
 参数：{'source_path': 'a.txt', 'destination_path': 'C:\\Users\\shkst\\Desktop\\a.txt'}


步骤2：调用对应的工具

In [12]:
if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args)  #调用工具
    print("工具执行的结果", result)

工具执行的结果 File moved successfully from a.txt to C:\Users\shkst\Desktop\a.txt.


## 如何把大模型解析出来的 tool_calls 变成真正的执行结果，并反馈给模型

这是一个非常好的问题。

在现代 LangChain 开发中，“如何把大模型解析出来的 tool_calls 变成真正的执行结果，并反馈给模型” 是最核心的环节。

虽然 LangChain 提供了 AgentExecutor 来自动做这件事，但为了让你彻底理解全流程，我将手动实现这个执行循环（Execution Loop）。这是掌握 Agent 原理的必经之路。

我们将使用一个自定义的计算工具和一个模拟的天气工具来演示一个完整的案例。

### 完整案例：手动执行 Tool Calls 闭环
这个案例演示了：用户问“北京天气怎么样？然后把气温乘以 2”，模型自动调用天气工具，拿到结果后，再调用计算工具，最后回答用户。

#### 1. 环境准备与工具定义

In [17]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# 加载环境变量
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义大模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- 步骤 1: 定义工具 (使用 @tool 装饰器) ---

@tool
def get_current_weather(city: str) -> str:
    """查询指定城市的当前天气情况。"""
    # 这里我们模拟一个 API 调用，实际项目中可以调用真实的天气 API
    print(f"--- [系统日志] 正在调用天气工具，查询城市: {city} ---")
    if "北京" in city:
        return "北京今天晴天，气温 25 度"
    elif "上海" in city:
        return "上海今天下雨，气温 20 度"
    else:
        return "未知城市天气"

@tool
def multiply(a: int, b: int) -> int:
    """计算两个数字的乘积。"""
    print(f"--- [系统日志] 正在调用乘法工具: {a} * {b} ---")
    return a * b

# 创建工具列表
tools = [get_current_weather, multiply]

# 创建工具映射表 (用于后面根据名字查找函数)
tools_map = {
    "get_current_weather": get_current_weather,
    "multiply": multiply
}

 #### 2. 绑定工具与初次调用

In [18]:
# --- 步骤 2: 绑定工具 ---
llm_with_tools = chat_model.bind_tools(tools)

# --- 步骤 3: 用户提问与初次模型调用 ---
query = "北京今天气温多少度？如果把这个度数乘以 10 是多少？"
messages = [HumanMessage(content=query)]

print(f"用户提问: {query}")
print("正在思考...")

# 第一次调用：模型只进行“决策”，不执行
ai_message = llm_with_tools.invoke(messages)

# 将模型的回复（包含工具调用请求）加入历史记录
messages.append(ai_message)

print(f"模型决策结果 (tool_calls): {ai_message.tool_calls}")

用户提问: 北京今天气温多少度？如果把这个度数乘以 10 是多少？
正在思考...
模型决策结果 (tool_calls): [{'name': 'get_current_weather', 'args': {'city': '北京'}, 'id': 'call_fVwxEUSZn4kux69XuQUPyMjd', 'type': 'tool_call'}]


#### 3. 核心环节：解析并执行工具 (Execution Loop)

这是你最关心的部分。我们需要遍历模型返回的 tool_calls，找到对应的函数并执行，然后封装成 ToolMessage。

In [19]:
# --- 步骤 4: 手动执行工具调用 ---

# 检查模型是否想要调用工具
if ai_message.tool_calls:
    print("\n>>> 开始执行工具...")

    # 遍历所有的工具调用请求 (模型可能一次性请求调用多个工具)
    for tool_call in ai_message.tool_calls:
        # 1. 获取工具名称和参数
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"] # 必须保留这个 ID，用于上下文匹配

        # 2. 从映射表中找到真正的工具函数
        selected_tool = tools_map.get(tool_name)

        if selected_tool:
            # 3. 执行工具！
            # invoke 会自动处理参数验证
            tool_result = selected_tool.invoke(tool_args)

            print(f"工具 '{tool_name}' 执行结果: {tool_result}")

            # 4. 创建 ToolMessage
            # 这是告诉模型“你刚才让我调用的工具，结果是这个”
            tool_message = ToolMessage(
                content=str(tool_result), # 结果必须转为字符串
                name=tool_name,
                tool_call_id=tool_call_id # 关键：ID 必须对应
            )

            # 5. 将执行结果加入消息历史
            messages.append(tool_message)
        else:
            print(f"错误：找不到工具 {tool_name}")

    print(">>> 工具执行完毕，结果已存入消息历史。\n")


>>> 开始执行工具...
--- [系统日志] 正在调用天气工具，查询城市: 北京 ---
工具 'get_current_weather' 执行结果: 北京今天晴天，气温 25 度
>>> 工具执行完毕，结果已存入消息历史。



#### 4. 二次调用：生成最终回答
现在，消息历史里包含了：[用户问题, 模型决策(tool_calls), 工具执行结果(ToolMessage)]。我们将这个完整的上下文再次发给模型。

In [23]:
# --- 步骤 5: 将工具结果反馈给模型，生成最终答案 ---

print("正在生成最终回答...")
final_response = llm_with_tools.invoke(messages)

print("-" * 30)
print(f"最终答案: {final_response}")
print("-" * 30)

正在生成最终回答...
------------------------------
最终答案: content='' additional_kwargs={'tool_calls': [{'id': 'call_8AABsdeWXeUZNxXi41vfR6Bn', 'function': {'arguments': '{"a":25,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 125, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--d292b60d-06fa-40fe-b8a0-3fc757cda1f8-0' tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 10}, 'id': 'call_8AABsdeWXeUZNxXi41vfR6Bn', 'type': 'tool_call'}] usage_metadata={'input_tokens': 125, 'output_tokens': 18, 'total_tokens': 143}
------------------------------


### 代码运行逻辑图解

为了帮你理解这个过程，数据流是这样的：
1. HumanMessage: "北京气温多少？乘以10是多少？"
⬇️ (发给 LLM)
2. AIMessage (tool_calls): [{name: get_current_weather, args: {city: 北京}}]
⬇️ (Python 捕获到 tool_calls，执行函数)
⬇️ (得到结果: "北京今天晴天，气温 25 度")
⬇️ (封装为 ToolMessage)
3. ToolMessage: content="北京...25度", tool_call_id=...
⬇️ (再次发给 LLM，此时历史记录里有上下文了)
4. AIMessage (tool_calls): [{name: multiply, args: {a: 25, b: 10}}]
(注：智能的模型可能分两步，先查天气，看到25度后，再发起第二次调用算乘法。也可能一次性规划好，取决于模型能力。在这个例子中，GPT-4o 往往非常聪明，甚至不需要调乘法工具直接口算，或者分步调用)
假设模型决定再次调用乘法工具：
⬇️ (Python 执行 multiply(25, 10))
⬇️ (得到结果: 250)
5. ToolMessage: content="250", tool_call_id=...
⬇️ (全部发给 LLM)
6. AIMessage (Final): "北京今天气温 25 度，乘以 10 后的结果是 250。"


### 关键点总结

1. ToolMessage 是桥梁：
你不能直接把 250 这个数字发给 LLM。你必须把它包装成 ToolMessage，并且带上 tool_call_id。这样 LLM 才知道“哦，这个 250 是我刚才要求调用的那个乘法函数返回的结果”。
2. Tools Map 很重要：
模型返回的只是一个字符串 "multiply"。你需要一个字典 {"multiply": multiply_func} 来把字符串映射到真正的 Python 函数上。
3. 循环机制：
在复杂的 Agent（如 ReAct Agent）中，上面的 步骤 3 ~ 步骤 5 会被放入一个 while 循环中。只要模型一直返回 tool_calls，程序就一直执行并反馈，直到模型不再调用工具，输出文本为止。这就是 AgentExecutor 的底层原理。

In [24]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# 1. 初始化
dotenv.load_dotenv()
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 定义工具
@tool
def get_current_weather(city: str) -> str:
    """查询天气"""
    if "北京" in city: return "25" # 为了方便计算，直接回数字字符串
    return "20"

@tool
def multiply(a: int, b: int) -> int:
    """乘法计算"""
    return a * b

tools = [get_current_weather, multiply]
tools_map = {t.name: t for t in tools} # 自动构建映射表
llm_with_tools = chat_model.bind_tools(tools)

# 3. 开始对话
query = "北京今天气温多少度？如果把这个度数乘以 10 是多少？"
messages = [HumanMessage(content=query)]

print(f"User: {query}\n")

# --- 核心循环逻辑 ---

# 第一次调用模型
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg) # 记得把模型的回复加入历史

# 只要模型还想调用工具 (tool_calls 不为空)，就一直循环
while ai_msg.tool_calls:
    print(f"🤖 模型想要调用工具: {len(ai_msg.tool_calls)} 个")

    for tool_call in ai_msg.tool_calls:
        # 1. 解析
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        print(f"   -> 调用 {tool_name} 参数: {tool_args}")

        # 2. 执行
        selected_tool = tools_map[tool_name]
        tool_result = selected_tool.invoke(tool_args)
        print(f"   <- 结果: {tool_result}")

        # 3. 封装消息
        messages.append(ToolMessage(
            content=str(tool_result),
            name=tool_name,
            tool_call_id=tool_call["id"]
        ))

    # 4. 把工具结果给模型，让它进行下一轮思考
    print("🔄 把结果反馈给模型，继续思考...\n")
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg) # 更新 ai_msg，用于下一次 while 判断

# --- 循环结束，输出最终结果 ---
print("-" * 30)
print(f"最终答案 (Content): {ai_msg.content}")
print("-" * 30)

User: 北京今天气温多少度？如果把这个度数乘以 10 是多少？

🤖 模型想要调用工具: 1 个
   -> 调用 get_current_weather 参数: {'city': '北京'}
   <- 结果: 25
🔄 把结果反馈给模型，继续思考...

🤖 模型想要调用工具: 1 个
   -> 调用 multiply 参数: {'a': 25, 'b': 10}
   <- 结果: 250
🔄 把结果反馈给模型，继续思考...

------------------------------
最终答案 (Content): 今天北京的气温是 25 度。如果把这个度数乘以 10，结果是 250。
------------------------------
